# Notebook 2: Prompt Engineering Aplicado a AWS Bedrock

En este notebook aprenderás a:
- Aplicar técnicas de prompting: zero-shot, few-shot y chain-of-thought
- Usar system prompts para controlar el comportamiento del modelo
- Comparar outputs de distintos modelos para el mismo prompt
- Ajustar parámetros de inferencia y medir su impacto

---

> **Requisito:** Haber completado el Notebook 1 o tener boto3 instalado y credenciales AWS configuradas.

## 1. Setup inicial

In [ ]:
# !pip install boto3 --quiet

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION=os.getenv("AWS_DEFAULT_REGION")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import boto3
import json
import time

REGION   = AWS_DEFAULT_REGION
MODEL_ID = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

def invocar_claude(prompt, system=None, temperature=0.7, max_tokens=1024):
    """Helper genérico para invocar Claude en Bedrock."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        body["system"] = system

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return resultado["content"][0]["text"]

print("✅ Setup listo")

✅ Setup listo


## 2. Zero-Shot Prompting

El modelo resuelve la tarea **sin ejemplos previos**, solo con instrucciones. Es el punto de partida más simple.

In [ ]:
# Clasificación de sentimiento en modo zero-shot
prompt_zero_shot = """
Clasifica el sentimiento del siguiente texto como POSITIVO, NEGATIVO o NEUTRO.
Responde únicamente con una de esas tres palabras.

Texto: "El servicio fue lento pero la comida estaba deliciosa."
"""

respuesta = invocar_claude(prompt_zero_shot, temperature=0.0)
print(f"Zero-Shot → {respuesta}")

Zero-Shot → POSITIVO


In [ ]:
# Probar con varios textos
textos = [
    "Me encantó la película, la recomendaría a todos.",
    "El producto llegó roto y tardó dos semanas.",
    "El paquete llegó hoy por la tarde.",
    "Increíble experiencia, superó todas mis expectativas."
]

print("Clasificación de sentimientos (Zero-Shot)")
print("-" * 50)
for texto in textos:
    prompt = f"""Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRO.
Responde solo con la etiqueta.

Texto: \"{texto}\""""
    sentimiento = invocar_claude(prompt, temperature=0.0)
    print(f"[{sentimiento.strip():8}] {texto}")

Clasificación de sentimientos (Zero-Shot)
--------------------------------------------------
[POSITIVO] Me encantó la película, la recomendaría a todos.
[NEGATIVO] El producto llegó roto y tardó dos semanas.
[NEUTRO  ] El paquete llegó hoy por la tarde.
[POSITIVO] Increíble experiencia, superó todas mis expectativas.


## 3. Few-Shot Prompting

Incluimos **ejemplos en el prompt** para guiar al modelo hacia el formato y comportamiento deseado. Muy útil para tareas específicas o formatos de salida personalizados.

In [ ]:
prompt_few_shot = """
Eres un extractor de información de reseñas de productos.
Dado un texto, extrae: producto, sentimiento (POS/NEG/NEU) y aspecto principal.
Responde en formato JSON.

### Ejemplos ###

Texto: "Las zapatillas son cómodísimas, perfectas para correr."
Respuesta: {"producto": "zapatillas", "sentimiento": "POS", "aspecto": "comodidad"}

Texto: "El laptop se calienta mucho después de una hora de uso."
Respuesta: {"producto": "laptop", "sentimiento": "NEG", "aspecto": "temperatura"}

Texto: "El auricular tiene 20 horas de batería según el manual."
Respuesta: {"producto": "auricular", "sentimiento": "NEU", "aspecto": "batería"}

### Tu turno ###

Texto: "La cafetera hace un ruido horrible pero el café sale excelente."
Respuesta:
"""

respuesta_fs = invocar_claude(prompt_few_shot, temperature=0.0)
print("Few-Shot → ", respuesta_fs)

Few-Shot →  ```json
{
  "producto": "cafetera",
  "sentimiento": "NEU",
  "aspecto": "ruido y calidad del café"
}
```

**Justificación:** La reseña contiene dos aspectos contradictorios: uno negativo (ruido horrible) y uno positivo (café excelente). Al haber sentimientos mixtos, se clasifica como NEU (neutral). El aspecto principal abarca tanto el ruido como la calidad del café, que son los dos puntos centrales mencionados.


In [ ]:
# Verificar que la respuesta es JSON válido
try:
    datos = json.loads(respuesta_fs.strip())
    print("✅ JSON válido:")
    for k, v in datos.items():
        print(f"  {k}: {v}")
except json.JSONDecodeError:
    print("⚠️  La respuesta no es JSON puro. Respuesta:", respuesta_fs)

⚠️  La respuesta no es JSON puro. Respuesta: ```json
{
  "producto": "cafetera",
  "sentimiento": "NEU",
  "aspecto": "ruido y calidad del café"
}
```

**Justificación:** La reseña contiene dos aspectos contradictorios: uno negativo (ruido horrible) y uno positivo (café excelente). Al haber sentimientos mixtos, se clasifica como NEU (neutral). El aspecto principal abarca tanto el ruido como la calidad del café, que son los dos puntos centrales mencionados.


## 4. Chain-of-Thought (CoT) Prompting

Pedimos al modelo que **razone paso a paso** antes de dar la respuesta final. Mejora significativamente la precisión en tareas de razonamiento, matemáticas y lógica.

In [ ]:
problema = """
Una tienda tiene 150 camisetas. Vende el 40% el lunes,
luego recibe un nuevo envío de 60 unidades el martes,
y el miércoles vende 35 unidades más.
¿Cuántas camisetas quedan al final del miércoles?
"""

# Sin Chain-of-Thought
print("=== Sin CoT ===")
prompt_sin_cot = f"{problema}\nResponde solo con el número final."
print(invocar_claude(prompt_sin_cot, temperature=0.0))

print()

# Con Chain-of-Thought
print("=== Con CoT ===")
prompt_con_cot = f"""{problema}
Piensa paso a paso antes de responder:
1. Calcula cuántas se vendieron el lunes
2. Suma el nuevo envío
3. Resta las ventas del miércoles
4. Da el resultado final
"""
print(invocar_claude(prompt_con_cot, temperature=0.0))

=== Sin CoT ===
127

=== Con CoT ===
# Solución paso a paso

## 1. Camisetas vendidas el lunes
- Se vende el 40% de 150
- 40% de 150 = 0.40 × 150 = **60 camisetas**
- Quedan: 150 - 60 = **90 camisetas**

## 2. Nuevo envío el martes
- Se reciben 60 unidades
- Total: 90 + 60 = **150 camisetas**

## 3. Ventas del miércoles
- Se venden 35 unidades
- Quedan: 150 - 35 = **115 camisetas**

## Resultado final
**Al final del miércoles quedan 115 camisetas**


## 5. System Prompts: Controlar el rol y comportamiento

El **system prompt** define el contexto, rol y restricciones del modelo antes de cualquier mensaje del usuario. Es una herramienta muy poderosa para personalizar el comportamiento.

In [ ]:
pregunta = "¿Qué es una API REST?"

sistemas = {
    "Experto técnico": "Eres un arquitecto de software senior. Responde de forma técnica y precisa, usando terminología específica del sector.",
    "Profesor para niños": "Eres un profesor de primaria. Explica los conceptos usando analogías simples y ejemplos de la vida cotidiana. Usa un lenguaje muy sencillo.",
    "Vendedor de producto": "Eres un comercial de soluciones tecnológicas. Destaca los beneficios de negocio y el valor que aporta la tecnología al cliente."
}

for rol, system_prompt in sistemas.items():
    print(f"\n{'='*60}")
    print(f"ROL: {rol}")
    print(f"{'='*60}")
    respuesta = invocar_claude(pregunta, system=system_prompt, temperature=0.5, max_tokens=300)
    print(respuesta)


ROL: Experto técnico
# API REST

Una **API REST** (Interfaz de Programación de Aplicaciones basada en Transferencia de Estado Representacional) es una arquitectura de servicios web que utiliza el protocolo HTTP para permitir la comunicación entre sistemas distribuidos.

## Principios Fundamentales

### 1. **Arquitectura Cliente-Servidor**
- Separación clara de responsabilidades
- Cliente realiza solicitudes; servidor proporciona recursos

### 2. **Operaciones HTTP Estándar**
```
GET     → Recuperar recursos
POST    → Crear nuevos recursos
PUT     → Actualizar recursos completos
PATCH   → Actualizar parcialmente
DELETE  → Eliminar recursos
HEAD    → Obtener metadatos
OPTIONS → Describir opciones de comunicación
```

### 3. **Recursos Identificables**
- Cada recurso tiene una URI única: `/api/v1/usuarios/123`
- Representación independiente del formato (JSON, XML, etc.)

### 4. **Statelessness**
- Cada solicitud contiene toda la información necesaria
- El servidor no almacena contexto de

## 6. Comparar modelos con el mismo prompt

Bedrock da acceso a múltiples modelos. Podemos comparar sus salidas para el mismo prompt.

In [ ]:
def invocar_modelo_generico(model_id, prompt, max_tokens=512):
    """Invoca distintos modelos adaptando el formato del body."""
    if "anthropic" in model_id:
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}]
        }
        response = bedrock_runtime.invoke_model(
            modelId=model_id, body=json.dumps(body),
            contentType="application/json", accept="application/json"
        )
        resultado = json.loads(response["body"].read())
        return resultado["content"][0]["text"]

    elif "amazon.titan" in model_id:
        body = {
            "inputText": prompt,
            "textGenerationConfig": {"maxTokenCount": max_tokens, "temperature": 0.7}
        }
        response = bedrock_runtime.invoke_model(
            modelId=model_id, body=json.dumps(body),
            contentType="application/json", accept="application/json"
        )
        resultado = json.loads(response["body"].read())
        return resultado["results"][0]["outputText"]

    elif "meta.llama" in model_id:
        body = {
            "prompt": prompt,
            "max_gen_len": max_tokens,
            "temperature": 0.7
        }
        response = bedrock_runtime.invoke_model(
            modelId=model_id, body=json.dumps(body),
            contentType="application/json", accept="application/json"
        )
        resultado = json.loads(response["body"].read())
        return resultado["generation"]

    return "Modelo no soportado en este helper"


# Modelos a comparar (ajusta según los que tengas habilitados)
modelos_comparar = [
    ("Claude 3 Haiku",  "eu.anthropic.claude-haiku-4-5-20251001-v1:0"),
    ("Llama 3.2 3B Instruct",    "eu.meta.llama3-2-3b-instruct-v1:0"),
]

prompt_comparacion = "Explica el concepto de 'overfitting' en machine learning en máximo 3 oraciones."

print(f"PROMPT: {prompt_comparacion}")
print("=" * 70)

for nombre, model_id in modelos_comparar:
    print(f"\n🤖 {nombre} ({model_id}):")
    print("-" * 50)
    try:
        respuesta = invocar_modelo_generico(model_id, prompt_comparacion)
        print(respuesta.strip())
    except Exception as e:
        print(f"  ⚠️  Error: {e}")
    time.sleep(1)  # Evitar throttling

PROMPT: Explica el concepto de 'overfitting' en machine learning en máximo 3 oraciones.

🤖 Claude 3 Haiku (eu.anthropic.claude-haiku-4-5-20251001-v1:0):
--------------------------------------------------
# Overfitting en Machine Learning

El **overfitting** ocurre cuando un modelo de aprendizaje automático se ajusta demasiado bien a los datos de entrenamiento, memorizando patrones específicos e incluso ruido, en lugar de aprender patrones generales. Esto causa que el modelo tenga excelente rendimiento en los datos de entrenamiento pero un rendimiento pobre en datos nuevos o de prueba. Es un problema común que se puede mitigar con técnicas como regularización, validación cruzada o usar menos características.

🤖 Nova 2 Lite (eu.amazon.nova-2-lite-v1:0):
--------------------------------------------------
Modelo no soportado en este helper


## 7. Impacto de los parámetros de inferencia

Veamos cómo `temperature` y `top_p` afectan la creatividad y variabilidad de las respuestas.

In [ ]:
prompt_creativo = "Escribe el primer párrafo de una historia de ciencia ficción."

configs = [
    {"temperature": 0.0, "label": "🧊 T=0.0 (determinista)"},
    {"temperature": 0.5, "label": "🌡️  T=0.5 (equilibrado)"},
    {"temperature": 1.0, "label": "🔥 T=1.0 (muy creativo)"},
]

for config in configs:
    print(f"\n{config['label']}")
    print("-" * 40)
    respuesta = invocar_claude(prompt_creativo, temperature=config["temperature"], max_tokens=200)
    print(respuesta.strip())
    time.sleep(0.5)


🧊 T=0.0 (determinista)
----------------------------------------
# El Último Transmissor

La lluvia ácida golpeaba contra la cúpula de cristal blindado mientras Maya observaba cómo los últimos satélites de comunicación se desintegraban en la atmósfera, dejando tras de sí rastros de fuego que iluminaban el cielo gris de Marte. Desde su puesto en la estación Kepler-7, a treinta mil kilómetros de la Tierra, sabía que aquello significaba una sola cosa: la humanidad acababa de perder su último hilo de conexión con el hogar, y ella era la única que podía intentar restaurarlo antes de que fuera demasiado tarde.

🌡️  T=0.5 (equilibrado)
----------------------------------------
# El Último Transmissor

La lluvia ácida golpeaba contra la cúpula de cristal de Nueva Babilonia mientras Dr. Kael observaba las lecturas del monitor: la magnetosfera terrestre se desintegraba más rápido de lo previsto. En tres días, quizás menos, la radiación solar aniquilaría toda forma de vida en la superficie. Pero K

In [ ]:
# Demostrar la determinismo de temperature=0
print("Verificando determinismo con temperature=0.0:")
prompt_corto = "Di exactamente 5 palabras sobre el espacio."

for i in range(3):
    r = invocar_claude(prompt_corto, temperature=0.0, max_tokens=50)
    print(f"  Intento {i+1}: {r.strip()}")
    time.sleep(0.5)

Verificando determinismo con temperature=0.0:
  Intento 1: El universo es infinitamente fascinante.
  Intento 2: El universo es infinitamente fascinante.
  Intento 3: El universo es infinitamente fascinante.


## 8. Prompt Templates reutilizables

Una buena práctica es crear templates parametrizados para no repetir código.

In [ ]:
class PromptTemplate:
    """Clase simple para gestionar templates de prompts."""

    def __init__(self, template: str):
        self.template = template

    def format(self, **kwargs) -> str:
        return self.template.format(**kwargs)


# Template de resumen
template_resumen = PromptTemplate("""
Resume el siguiente texto en {num_puntos} puntos clave.
Usa un tono {tono}.
Formato: lista numerada.

TEXTO:
{texto}

RESUMEN:
""")

texto_ejemplo = """
AWS Bedrock es un servicio completamente gestionado que ofrece una selección de modelos
de fundación de alto rendimiento de las principales empresas de IA como Anthropic,
Cohere, Meta, Mistral y Amazon, a través de una única API. Bedrock permite construir
aplicaciones de IA generativa de forma segura, privada y responsable sin necesidad de
gestionar infraestructura. Los datos de los clientes nunca se usan para entrenar los
modelos base y se pueden aplicar controles de seguridad a nivel empresarial.
"""

prompt_final = template_resumen.format(
    num_puntos=3,
    tono="formal y conciso",
    texto=texto_ejemplo
)

print("=== Prompt generado ===")
print(prompt_final)
print("\n=== Respuesta ===")
print(invocar_claude(prompt_final, temperature=0.3))

=== Prompt generado ===

Resume el siguiente texto en 3 puntos clave.
Usa un tono formal y conciso.
Formato: lista numerada.

TEXTO:

AWS Bedrock es un servicio completamente gestionado que ofrece una selección de modelos
de fundación de alto rendimiento de las principales empresas de IA como Anthropic,
Cohere, Meta, Mistral y Amazon, a través de una única API. Bedrock permite construir
aplicaciones de IA generativa de forma segura, privada y responsable sin necesidad de
gestionar infraestructura. Los datos de los clientes nunca se usan para entrenar los
modelos base y se pueden aplicar controles de seguridad a nivel empresarial.


RESUMEN:


=== Respuesta ===
# Resumen: AWS Bedrock

1. **Servicio gestionado con múltiples modelos**: AWS Bedrock proporciona acceso a modelos de fundación de alto rendimiento de empresas líderes en IA (Anthropic, Cohere, Meta, Mistral y Amazon) mediante una única API integrada.

2. **Desarrollo seguro sin gestión de infraestructura**: Permite construir apl

## 9. Buenas prácticas de Prompt Engineering

| Técnica | Cuándo usarla | Ejemplo |
|--------|--------------|--------|
| **Zero-shot** | Tareas simples y directas | Clasificar, traducir, resumir |
| **Few-shot** | Formatos de salida específicos | JSON estructurado, tablas |
| **Chain-of-Thought** | Razonamiento complejo | Matemáticas, lógica, análisis |
| **System Prompt** | Definir rol y restricciones | Chatbots especializados |
| **Temperature baja** | Respuestas consistentes | Datos, código, clasificación |
| **Temperature alta** | Creatividad y variedad | Brainstorming, escritura creativa |

---

**Siguiente notebook →** Embeddings y Búsqueda Semántica con Bedrock